# SBA 7(a) Loan Risk — Logistic Regression Baseline

This notebook develops a baseline Logistic Regression model
to predict whether an SBA 7(a) loan experiences a recorded
charge-off within five calendar years of approval.

Model development uses chronological approval-year cohorts
to reduce temporal leakage.

- Training: FY2010–2015
- Hyperparameter tuning: FY2016
- Probability calibration: FY2017
- Independent validation: FY2018
- Final holdout test: FY2019

> **Reproducibility note:** This notebook was originally executed from the project `notebooks/` directory using Python 3.12.8 and scikit-learn 1.5.1. The source CSV (`../data/exports/sba_model_export.csv`) is not distributed in GitHub because it exceeds the normal file-size limit. Generate it using the documented PostgreSQL export workflow before executing cells. Saved outputs below are historical results; editing documentation does not constitute a new execution.


In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

pd.set_option("display.max_columns", None)

RANDOM_STATE = 42

DATA_PATH = Path("../data/exports/sba_model_export.csv")

if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Missing modeling CSV: {DATA_PATH}. Generate it via the PostgreSQL export and run from notebooks/."
    )

In [2]:
df = pd.read_csv(
    DATA_PATH,
    parse_dates=["approval_date"],
    dtype={
        "sba_lender_id": "string",
        "lender_key": "string"
    }
)

print("Dataset shape:", df.shape)

assert df.shape == (478953, 25)

Dataset shape: (478953, 25)


> **Historical baseline (15 features):** This notebook reproduces the initial candidate-feature Logistic Regression model, including loan term. The term feature and its validity flag were later excluded after leakage investigation in Notebook 03. Do not interpret this artifact as the final selected 13-feature risk model.


In [3]:
numeric_features = [
    "gross_approval",
    "guarantee_pct",
    "term_months_model",
    "initial_interest_rate_model",
    "jobs_supported"
]

categorical_features = [
    "business_age_group",
    "naics_sector",
    "project_state",
    "processing_method",
    "collateral_indicator",
    "revolver_status",
    "interest_rate_type",
    "business_type"
]

flag_features = [
    "invalid_term_flag",
    "invalid_interest_rate_flag"
]

feature_columns = (
    numeric_features
    + categorical_features
    + flag_features
)

target_column = "chargeoff_within_5yr"

print("Number of candidate predictors:", len(feature_columns))
print("Target variable:", target_column)

Number of candidate predictors: 15
Target variable: chargeoff_within_5yr


In [4]:
train_df = df.loc[
    df["modeling_period"] == "Train"
].copy()

tune_df = df.loc[
    df["modeling_period"] == "Tune"
].copy()

X_train = train_df[feature_columns]
y_train = train_df[target_column]

X_tune = tune_df[feature_columns]
y_tune = tune_df[target_column]

print("Training:", X_train.shape, y_train.shape)
print("Tuning:", X_tune.shape, y_tune.shape)

assert len(X_train) == 266226
assert len(X_tune) == 56785

Training: (266226, 15) (266226,)
Tuning: (56785, 15) (56785,)


In [5]:
from sklearn.impute import SimpleImputer

numeric_imputer = SimpleImputer(
    strategy="median"
)

numeric_imputer.fit(
    X_train[numeric_features]
)

print("Training medians used for imputation:")

for feature, median in zip(
    numeric_features,
    numeric_imputer.statistics_
):
    print(f"{feature}: {median:.4f}")

Training medians used for imputation:
gross_approval: 113700.0000
guarantee_pct: 75.0000
term_months_model: 84.0000
initial_interest_rate_model: 6.0000
jobs_supported: 5.0000


In [6]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

print(numeric_transformer)

Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())])


In [7]:
from sklearn.preprocessing import OneHotEncoder

categorical_transformer = Pipeline(
    steps=[
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                min_frequency=100
            )
        )
    ]
)

print(categorical_transformer)

Pipeline(steps=[('encoder',
                 OneHotEncoder(handle_unknown='ignore', min_frequency=100))])


In [8]:
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
        ("flags", "passthrough", flag_features)
    ],
    remainder="drop"
)

print(preprocessor)

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['gross_approval', 'guarantee_pct',
                                  'term_months_model',
                                  'initial_interest_rate_model',
                                  'jobs_supported']),
                                ('cat',
                                 Pipeline(steps=[('encoder',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                min_frequency=100))]),
                                 ['business_age_group', 'naics_sector',
                                  'project_state', 'processing_method',
                                  'collateral_indicator', 'revolver_s

In [9]:
from sklearn.linear_model import LogisticRegression

logistic_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                C=1.0,
                penalty="l2",
                solver="lbfgs",
                max_iter=2000,
                random_state=RANDOM_STATE
            )
        )
    ]
)

print(logistic_pipeline)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['gross_approval',
                                                   'guarantee_pct',
                                                   'term_months_model',
                                                   'initial_interest_rate_model',
                                                   'jobs_supported']),
                                                 ('cat',
                                                  Pipeline(steps=[('encoder',
                                                                   O

In [10]:
import warnings
from sklearn.exceptions import ConvergenceWarning

with warnings.catch_warnings(record=True) as caught_warnings:
    warnings.simplefilter("always", ConvergenceWarning)

    logistic_pipeline.fit(X_train, y_train)

convergence_warnings = [
    warning
    for warning in caught_warnings
    if issubclass(warning.category, ConvergenceWarning)
]

print("Training completed.")
print("Convergence warnings:", len(convergence_warnings))

classifier = logistic_pipeline.named_steps["classifier"]

print("Iterations used:", classifier.n_iter_)

Training completed.
Convergence warnings: 0
Iterations used: [45]


In [11]:
# Retrieve fitted preprocessing and classification components
fitted_preprocessor = logistic_pipeline.named_steps["preprocessor"]
fitted_classifier = logistic_pipeline.named_steps["classifier"]

# Get feature names after preprocessing
transformed_feature_names = (
    fitted_preprocessor.get_feature_names_out()
)

# Count the coefficients learned by Logistic Regression
coefficient_count = fitted_classifier.coef_.shape[1]

print("Original predictor columns:", len(feature_columns))
print("Transformed feature columns:", len(transformed_feature_names))
print("Model coefficients:", coefficient_count)
print("Model intercepts:", fitted_classifier.intercept_.shape[0])

# Validate feature and coefficient alignment
assert len(transformed_feature_names) == coefficient_count
assert np.isfinite(fitted_classifier.coef_).all()
assert np.isfinite(fitted_classifier.intercept_).all()

print("\nFeature and coefficient validation passed!")

# Display the first 15 transformed feature names
print("\nFirst 15 transformed features:")

for feature in transformed_feature_names[:15]:
    print(feature)

Original predictor columns: 15
Transformed feature columns: 116
Model coefficients: 116
Model intercepts: 1

Feature and coefficient validation passed!

First 15 transformed features:
num__gross_approval
num__guarantee_pct
num__term_months_model
num__initial_interest_rate_model
num__jobs_supported
cat__business_age_group_2–3 Years
cat__business_age_group_3–4 Years
cat__business_age_group_4–5 Years
cat__business_age_group_5+ Years
cat__business_age_group_Startup
cat__business_age_group_Under 1 Year
cat__business_age_group_Unknown
cat__business_age_group_infrequent_sklearn
cat__naics_sector_Accommodation and Food Services
cat__naics_sector_Administrative and Support Services


In [12]:
# Generate charge-off probabilities for FY2016
tune_probabilities = logistic_pipeline.predict_proba(X_tune)[:, 1]

# Check probability output
print("Number of predictions:", len(tune_probabilities))
print("Minimum probability:", tune_probabilities.min())
print("Maximum probability:", tune_probabilities.max())
print("Average predicted probability:", tune_probabilities.mean())

# Validate prediction output
assert len(tune_probabilities) == len(y_tune)
assert np.isfinite(tune_probabilities).all()
assert ((tune_probabilities >= 0) & (tune_probabilities <= 1)).all()

print("\nProbability validation passed!")

Number of predictions: 56785
Minimum probability: 4.889199637634178e-06
Maximum probability: 0.6428125190301573
Average predicted probability: 0.03497649557358029

Probability validation passed!


In [13]:
from sklearn.metrics import average_precision_score, roc_auc_score

# Actual charge-off rate in the FY2016 tuning cohort
tune_prevalence = y_tune.mean()

# Average Precision summarizes precision-recall performance
tune_ap = average_precision_score(
    y_tune,
    tune_probabilities
)

# Compare model performance against event prevalence
tune_ap_lift = tune_ap / tune_prevalence

# ROC-AUC provides a secondary ranking diagnostic
tune_roc_auc = roc_auc_score(
    y_tune,
    tune_probabilities
)

print(f"FY2016 event prevalence: {tune_prevalence:.4%}")
print(f"Logistic Regression Average Precision: {tune_ap:.4f}")
print(f"Average Precision lift: {tune_ap_lift:.2f}x")
print(f"ROC-AUC: {tune_roc_auc:.4f}")

FY2016 event prevalence: 3.8707%
Logistic Regression Average Precision: 0.1550
Average Precision lift: 4.00x
ROC-AUC: 0.8482


In [14]:
# Create a table of FY2016 predictions and actual outcomes
tune_results = pd.DataFrame({
    "actual_chargeoff": y_tune.to_numpy(),
    "predicted_probability": tune_probabilities
})

# Define review capacity
review_fraction = 0.10

# Round up to ensure sufficient review capacity
review_count = int(np.ceil(
    len(tune_results) * review_fraction
))

# Select the highest-risk loans
top_risk_loans = tune_results.nlargest(
    review_count,
    "predicted_probability"
)

print("Total FY2016 loans:", len(tune_results))
print("Review capacity:", f"{review_fraction:.0%}")
print("Loans selected for review:", len(top_risk_loans))
print("Total actual charge-offs:", tune_results["actual_chargeoff"].sum())
print("Charge-offs in review group:", top_risk_loans["actual_chargeoff"].sum())

Total FY2016 loans: 56785
Review capacity: 10%
Loans selected for review: 5679
Total actual charge-offs: 2198
Charge-offs in review group: 1001


In [15]:
# Count actual charge-offs
total_chargeoffs = tune_results["actual_chargeoff"].sum()

# Count charge-offs captured in the top-risk review group
captured_chargeoffs = top_risk_loans["actual_chargeoff"].sum()

# Calculate operational metrics
recall_at_10 = captured_chargeoffs / total_chargeoffs

precision_at_10 = captured_chargeoffs / review_count

precision_lift_at_10 = precision_at_10 / tune_prevalence

# Calculate capture relative to a random-review reference
random_expected_chargeoffs = total_chargeoffs * (
    review_count / len(tune_results)
)

print(f"Recall at Top 10%: {recall_at_10:.2%}")
print(f"Precision at Top 10%: {precision_at_10:.2%}")
print(f"Precision lift at Top 10%: {precision_lift_at_10:.2f}x")
print(f"Expected charge-offs under random review: {random_expected_chargeoffs:.1f}")

# Validate metric calculations
assert 0 <= recall_at_10 <= 1
assert 0 <= precision_at_10 <= 1

print("\nOperational metric validation passed!")

Recall at Top 10%: 45.54%
Precision at Top 10%: 17.63%
Precision lift at Top 10%: 4.55x
Expected charge-offs under random review: 219.8

Operational metric validation passed!


In [16]:
# Candidate Logistic Regression regularization values
c_values = [0.1, 1.0, 10.0]

# Store tuning results after each model evaluation
tuning_results = []

print("Logistic Regression hyperparameters to evaluate:")

for c in c_values:
    print(f"C = {c}")

print("\nNumber of configurations:", len(c_values))

Logistic Regression hyperparameters to evaluate:
C = 0.1
C = 1.0
C = 10.0

Number of configurations: 3


In [17]:
from sklearn.base import clone

# Use the same top-10% capacity for every configuration
review_count = int(np.ceil(len(y_tune) * 0.10))

tuning_results = []
tuned_models = {}

for c in c_values:

    # Create a fresh, unfitted pipeline
    model = clone(logistic_pipeline)

    # Set the regularization parameter
    model.set_params(classifier__C=c)

    # Fit preprocessing and Logistic Regression on training data
    model.fit(X_train, y_train)

    # Predict charge-off probabilities for FY2016
    probabilities = model.predict_proba(X_tune)[:, 1]

    # Calculate Average Precision
    ap = average_precision_score(
        y_tune,
        probabilities
    )

    # Find the highest-scored loans
    top_indices = np.argsort(-probabilities, kind="stable")[:review_count]

    # Count actual charge-offs among selected loans
    captured_chargeoffs = y_tune.to_numpy()[top_indices].sum()

    # Calculate operational metrics
    recall_10 = captured_chargeoffs / y_tune.sum()
    precision_10 = captured_chargeoffs / review_count

    # Check convergence
    iterations = model.named_steps["classifier"].n_iter_[0]

    # Store evaluation metrics
    tuning_results.append({
        "C": c,
        "Average_Precision": ap,
        "Recall_at_10": recall_10,
        "Precision_at_10": precision_10,
        "Captured_Chargeoffs": int(captured_chargeoffs),
        "Iterations": int(iterations)
    })

    # Keep fitted models for later selection
    tuned_models[c] = model

    print(f"Completed C = {c}")

# Create a comparison table
tuning_df = pd.DataFrame(tuning_results)

print("\nLogistic Regression Tuning Results:")
display(tuning_df.round(4))

Completed C = 0.1
Completed C = 1.0
Completed C = 10.0

Logistic Regression Tuning Results:


,C,Average_Precision,Recall_at_10,Precision_at_10,Captured_Chargeoffs,Iterations
0,0.1,0.1542,0.4463,0.1727,981,33
1,1.0,0.1550,0.4554,0.1763,1001,45
2,10.0,0.1549,0.4568,0.1768,1004,45


In [18]:
# Select the model with the highest FY2016 Average Precision
best_result = tuning_df.loc[
    tuning_df["Average_Precision"].idxmax()
]

best_c = float(best_result["C"])

# Retrieve the already-fitted model
selected_logistic_model = tuned_models[best_c]

print("Selected C:", best_c)
print(f"Tuning Average Precision: {best_result['Average_Precision']:.4f}")
print(f"Tuning Recall@10%: {best_result['Recall_at_10']:.2%}")
print(f"Tuning Precision@10%: {best_result['Precision_at_10']:.2%}")

# Verify the selected model configuration
assert best_c == 1.0
assert selected_logistic_model.named_steps["classifier"].C == best_c

print("\nBaseline model selection validated!")

Selected C: 1.0
Tuning Average Precision: 0.1550
Tuning Recall@10%: 45.54%
Tuning Precision@10%: 17.63%

Baseline model selection validated!


In [19]:
import joblib
from pathlib import Path

# Create the models directory if it doesn't exist
MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Define the saved model location
model_path = MODEL_DIR / "logistic_regression_baseline.joblib"

# Save the fitted Logistic Regression pipeline
joblib.dump(selected_logistic_model, model_path)

print("Model saved to:", model_path)
print("File exists:", model_path.exists())

Model saved to: ../models/logistic_regression_baseline.joblib
File exists: True
